# Project : 서울시 빅데이터 활용 경진대회
- 프로젝트 목적 : 서울시 공공 데이터를 바탕으로 N년간의 물품 가격의 시계열 패턴, 지역별 차이, 외부 요인에 의한 영향 등을 정량화하고 시민 체감형 물가 현황 대시보드를 제작
- 데이터 정보
    - 규모 : 319,296 rows * 19 columns
    - 기간 : 2023-01-20 ~ 2026-04-14
    - 출처 : [서울 데이터 허브](https://data.seoul.go.kr/bsp/wgs/dataView/data300View/10003.do)
- 최종 수정 : 2026-05-01

## 라이브러리 import

In [ ]:
# 데이터 처리
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import stats
from datetime import datetime
import re
import math

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
# import missingno as msno  # 결측치 분포 시각화
## 한글 깨짐 방지
# 윈도우
# plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['axes.unicode_minus'] = False
# 맥
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 판다스 출력 옵션 설정 (데이터가 많을 때 생략되는 것을 방지)
pd.set_option('display.max_columns', None)  # 모든 컬럼 출력
pd.set_option('display.float_format', '{:.4f}'.format) # 소수점 4자리까지 고정 (가독성 향상)

# 그 외
# 경고 무시
import warnings
warnings.filterwarnings("ignore")

# 시스템
import os
import gc

## 데이터 로드 & 탐색

In [ ]:
PATH = '/Users/hyun/Documents/project/seoul-market-price-analysis/data/raw/'

df25 = pd.read_csv(f'{PATH}생필품 농수축산물 가격 정보(2025년이후).csv', header=1, low_memory=False) # 모든 데이터를 문자로 읽어옴, 메모리 경고를 끄고 한 번에 읽어서 타입을 추론
df24 = pd.read_csv(f'{PATH}생필품 농수축산물 가격 정보(2024년).csv', low_memory=False, encoding='cp949')
df23 = pd.read_csv(f'{PATH}생필품 농수축산물 가격 정보(2023년).csv', low_memory=False, encoding='cp949')

df25.shape, df24.shape, df23.shape

((144684, 19), (98053, 14), (76559, 14))

## preprocessing & feature engineering

### 칼럼명 통일

In [ ]:
col_mapping = {
    '일련번호':                         'sn',
    '시장/마트 번호':                   'mkplc_mart_no',
    '시장/마트 이름':                   'mkplc_mart_nm',
    '품목 번호':                        'prdlst_no',
    '품목 이름':                        'prdlst_nm',
    '실판매규격':                       'real_sle_stndrd',
    '가격(원)':                         'pc',
    '년도-월':                          'ym',
    '비고':                             'rmrk',
    '시장유형 구분(시장/마트) 코드':    'mkplc_type_cd',
    '시장유형 구분(시장/마트) 이름':    'mkplc_type_nm',
    '자치구 코드':                      'atdrc_cd',
    '자치구 이름':                      'atdrc',
    '점검일자':                         'chck_ymd',
}

df23.rename(columns=col_mapping, inplace=True)
df24.rename(columns=col_mapping, inplace=True)

print("✅ 칼럼명 통일 완료")

### 칼럼별 데이터타입 통일

In [ ]:
# ym 칼럼 포맷 통일
# df23 : '2023-02' 형식
df23['ym'] = pd.to_datetime(df23['ym'], format='%Y-%m', errors='coerce').dt.strftime('%Y-%m')
# df24 : 'Jan-24' 형식
df24['ym'] = pd.to_datetime(df24['ym'], format='%b-%y', errors='coerce').dt.strftime('%Y-%m')
# df25 : '2025-01' 형식
df25['ym'] = pd.to_datetime(df25['ym'], format='%Y-%m', errors='coerce').dt.strftime('%Y-%m')

# 2023-01 제거 (98건, 2월 누락으로 연속성 없음)
df23 = df23[df23['ym'] != '2023-01'].copy()

# mkplc_type_cd 타입 통일
for df in [df23, df24, df25]:
    df['mkplc_type_cd'] = df['mkplc_type_cd'].astype('Int64')

print("✅ ym 포맷 통일 완료")
print(f"  df23: {df23['ym'].min()} ~ {df23['ym'].max()}")
print(f"  df24: {df24['ym'].min()} ~ {df24['ym'].max()}")
print(f"  df25: {df25['ym'].min()} ~ {df25['ym'].max()}")

### 품목명 파싱으로 대표품목명 변수 생성

In [ ]:
def parse_item_details(text):
    """prdlst_nm → (핵심품목명, 품종, 규격) 분해"""
    if pd.isna(text):
        return '기타', '기본', '규격없음'

    text = str(text)

    # 품종/산지: 괄호 안 텍스트
    v_match = re.search(r'\((.*?)\)', text)
    variety = v_match.group(1).strip() if v_match else '기본'

    # 규격: 숫자 + 단위
    s_match = re.search(r'(\d+(?:\.\d+)?\s*[a-zA-Z가-힣]+)', text)
    spec = s_match.group(1).strip() if s_match else '규격없음'

    # 핵심 품목명: 괄호·숫자·단위·특수문자 제거
    core = re.sub(r'\(.*?\)', '', text)
    core = re.sub(r'\d+(?:\.\d+)?\s*[a-zA-Z가-힣]*', '', core)
    core = re.sub(r'[.,/]', '', core).strip()
    core = ' '.join(core.split())

    return core, variety, spec

for name, df in [('df23', df23), ('df24', df24), ('df25', df25)]:
    parsed = df['prdlst_nm'].apply(parse_item_details)
    df['new_std_nm'] = [x[0] for x in parsed]
    df['variety']    = [x[1] for x in parsed]
    df['spec']       = [x[2] for x in parsed]
    print(f"  {name} 파싱 완료 → 고유 품목 수: {df['new_std_nm'].nunique()}개")

print("✅ 품목명 파싱 완료")

### 품목번호 맵핑오류 처리

In [ ]:
# # 임계값 설정
# THRESHOLD = 0.05  # 5% 미만이면 오입력으로 판단

# def fix_mapping_errors(df, year_label, threshold=THRESHOLD):
#     df = df.copy()

#     # 결측치 처리 ('기타' → prdlst_no 기준 최빈 new_std_nm으로 대체)
#     mode_map = (
#         df[df['new_std_nm'] != '기타']
#         .groupby('prdlst_no')['new_std_nm']
#         .agg(lambda x: x.value_counts().index[0])
#         .to_dict()
#     )

#     null_before = (df['new_std_nm'] == '기타').sum()
#     df['new_std_nm'] = df.apply(
#         lambda row: mode_map.get(row['prdlst_no'], '기타')
#         if row['new_std_nm'] == '기타' else row['new_std_nm'],
#         axis=1
#     )
#     null_after = (df['new_std_nm'] == '기타').sum()
#     print(f"\n[{year_label}]")
#     print(f"  결측치(기타) 처리: {null_before:,}건 → {null_after:,}건")

#     # 오입력 탐지
#     total_counts  = df.groupby('prdlst_no')['new_std_nm'].transform('count')
#     mode_per_code = df.groupby('prdlst_no')['new_std_nm'].transform(
#         lambda x: x.value_counts().index[0]
#     )
#     item_counts = df.groupby(['prdlst_no', 'new_std_nm'])['new_std_nm'].transform('count')

#     df['_ratio']     = item_counts / total_counts
#     df['_mode_item'] = mode_per_code
#     df['_is_wrong']  = (
#         (df['new_std_nm'] != df['_mode_item']) &
#         (df['_ratio'] < threshold)
#     )

#     # 소멸 품목 보호
#     wrong_items   = set(df[df['_is_wrong']]['new_std_nm'])
#     correct_items = set(df[~df['_is_wrong']]['new_std_nm'])
#     vanishing     = wrong_items - correct_items

#     if vanishing:
#         print(f"  소멸 보호 품목: {sorted(vanishing)}")
#         df['_is_wrong'] = df['_is_wrong'] & ~df['new_std_nm'].isin(vanishing)

#     # drop 목록 출력 (확인용)
#     wrong_summary = (
#         df[df['_is_wrong']]
#         .groupby(['prdlst_no', '_mode_item', 'new_std_nm'])
#         .size()
#         .reset_index(name='건수')
#         .rename(columns={'_mode_item': '정답품목', 'new_std_nm': '오입력품목'})
#         .sort_values(['prdlst_no', '건수'], ascending=[True, False])
#     )
#     print(f"  drop 예정: {df['_is_wrong'].sum():,}건 ({len(wrong_summary)}종)")
#     if not wrong_summary.empty:
#         print(wrong_summary.to_string(index=False))

#     # drop 실행 및 임시 칼럼 정리
#     df = df[~df['_is_wrong']].copy()
#     df.drop(columns=['_ratio', '_mode_item', '_is_wrong'], inplace=True)
#     print(f"  처리 후: {len(df):,}행")

#     return df

# df23 = fix_mapping_errors(df23, 'df23')
# df24 = fix_mapping_errors(df24, 'df24')
# df25 = fix_mapping_errors(df25, 'df25')

# print("\n✅ 맵핑 오류 처리 완료")

In [ ]:
# 임계값 설정
THRESHOLD = 0.05  # 5% 미만이면 오입력으로 판단

def detect_mapping_errors(df, year_label, threshold=THRESHOLD):
    df = df.copy()

    # 결측치 처리 ('기타' → prdlst_no 기준 최빈 new_std_nm으로 대체)
    mode_map = (
        df[df['new_std_nm'] != '기타']
        .groupby('prdlst_no')['new_std_nm']
        .agg(lambda x: x.value_counts().index[0])
        .to_dict()
    )

    null_before = (df['new_std_nm'] == '기타').sum()
    df['new_std_nm'] = df.apply(
        lambda row: mode_map.get(row['prdlst_no'], '기타')
        if row['new_std_nm'] == '기타' else row['new_std_nm'],
        axis=1
    )
    null_after = (df['new_std_nm'] == '기타').sum()
    print(f"\n[{year_label}]")
    print(f"  결측치(기타) 처리: {null_before:,}건 → {null_after:,}건")

    # 오입력 탐지
    total_counts  = df.groupby('prdlst_no')['new_std_nm'].transform('count')
    mode_per_code = df.groupby('prdlst_no')['new_std_nm'].transform(
        lambda x: x.value_counts().index[0]
    )
    item_counts = df.groupby(['prdlst_no', 'new_std_nm'])['new_std_nm'].transform('count')

    df['_ratio']     = item_counts / total_counts
    df['_mode_item'] = mode_per_code
    df['_is_wrong']  = (
        (df['new_std_nm'] != df['_mode_item']) &
        (df['_ratio'] < threshold)
    )

    # 소멸 품목 보호
    wrong_items   = set(df[df['_is_wrong']]['new_std_nm'])
    correct_items = set(df[~df['_is_wrong']]['new_std_nm'])
    vanishing     = wrong_items - correct_items

    if vanishing:
        print(f"  소멸 보호 품목: {sorted(vanishing)}")
        df['_is_wrong'] = df['_is_wrong'] & ~df['new_std_nm'].isin(vanishing)

    # drop 목록 출력 (확인용)
    wrong_summary = (
        df[df['_is_wrong']]
        .groupby(['prdlst_no', '_mode_item', 'new_std_nm'])
        .size()
        .reset_index(name='건수')
        .rename(columns={'_mode_item': '정답품목', 'new_std_nm': '오입력품목'})
        .sort_values(['prdlst_no', '건수'], ascending=[True, False])
    )
    print(f"  drop 예정: {df['_is_wrong'].sum():,}건 ({len(wrong_summary)}종)")
    if not wrong_summary.empty:
        print(wrong_summary.to_string(index=False))

    return df  # _is_wrong 칼럼 포함한 상태로 반환 (drop 미실행)

df23 = detect_mapping_errors(df23, 'df23')
df24 = detect_mapping_errors(df24, 'df24')
df25 = detect_mapping_errors(df25, 'df25')

print("\n👆 위 목록 확인 후 drop 진행.")

In [ ]:
# 맵핑 오류 목록 drop
def apply_drop(df, year_label):
    df = df[~df['_is_wrong']].copy()
    df.drop(columns=['_ratio', '_mode_item', '_is_wrong'], inplace=True)
    print(f"[{year_label}] 처리 후: {len(df):,}행")
    return df

df23 = apply_drop(df23, 'df23')
df24 = apply_drop(df24, 'df24')
df25 = apply_drop(df25, 'df25')

print("\n✅ 맵핑 오류 처리 완료")

### 데이터 통합

In [ ]:
df_total = pd.concat([df23, df24, df25], axis=0, ignore_index=True)

print(f"✅ 통합 완료: {df_total.shape[0]:,}행 × {df_total.shape[1]}열")
del df23, df24, df25
gc.collect()
print("   원본 데이터프레임 메모리 해제")

### 전통시장 데이터만 필터링

- 대형마트 18,807 건으로 불균형이 너무 심해서 전통시장 vs 대형마트 비교보다 전통시장 물가 현황 방향으로 진행

In [ ]:
df_total = df_total[df_total['mkplc_type_nm'] == '전통시장'].copy()
df_total['chck_ymd'] = pd.to_datetime(df_total['chck_ymd'])
df_total = df_total.sort_values('chck_ymd').reset_index(drop=True)

print(f"✅ 전통시장 필터링 후: {len(df_total):,}행")

### 시계열 파생변수 생성

In [ ]:
# 시계열 파생변수 (반기 / 분기 / 계절)
def get_season(month):
    if month in [3, 4, 5]:    return '봄'
    elif month in [6, 7, 8]:  return '여름'
    elif month in [9, 10, 11]: return '가을'
    else:                      return '겨울'

df_total['반기'] = df_total['chck_ymd'].dt.month.apply(lambda m: '상반기' if m <= 6 else '하반기')
df_total['분기'] = df_total['chck_ymd'].dt.quarter
df_total['계절'] = df_total['chck_ymd'].dt.month.apply(get_season)

print("✅ 시계열 파생변수 생성 완료")

### 가격 관련 전처리

#### 보정 가격 계산(특정 품목)

In [ ]:
# 특정 품목 대표 규격 단위로 가격 보정
def calculate_standard_price(row):
    """규격별 가격을 단일 기준 단위 가격으로 보정"""
    item    = row['new_std_nm']
    variety = row['variety']
    spec    = row['spec']
    price   = row['pc']

    detailed_nm = f"{item}_{variety}" if variety != '기본' else item

    num_match = re.search(r'(\d+(?:\.\d+)?)', spec)
    amount = float(num_match.group(1)) if num_match else 1.0

    adj_price = price
    is_valid  = True

    if item == '쌀':
        adj_price = (price / amount) * 10 if 'kg' in spec else price
        is_valid  = 'kg' in spec

    elif item in ['소고기', '돼지고기', '닭고기', '감자']:
        if 'kg' in spec:
            adj_price = (price / (amount * 1000)) * 100
        elif 'g' in spec:
            adj_price = (price / amount) * 100
        else:
            is_valid = False

    elif item in ['고등어', '갈치', '조기']:
        if '손' in spec:
            adj_price = price / (amount * 2)
        elif '마리' in spec:
            adj_price = price / amount
        else:
            is_valid = False

    elif item == '포도':
        adj_price = (price / amount) * 2 if 'kg' in spec else price
        is_valid  = 'kg' in spec

    return detailed_nm, adj_price, is_valid

results = df_total.apply(calculate_standard_price, axis=1)
df_total['detailed_nm'] = [r[0] for r in results]
df_total['adj_price']   = [r[1] for r in results]
df_total['is_valid']    = [r[2] for r in results]

df_final = df_total[df_total['is_valid']].copy()
print(f"✅ 가격 보정 완료 → 유효 데이터: {len(df_final):,}행")

#### 가격 이상치 처리

In [ ]:
before = len(df_final)
df_clean = df_final[df_final['adj_price'] > 0].dropna(subset=['adj_price']).copy()

print(f"✅ 0 이하 제거: {before:,}건 → {len(df_clean):,}건  (제거: {before - len(df_clean):,}건)")

In [ ]:
# 휴먼 에러로 판단되는 이상치 보정
def auto_correct_price_typos(df):
    """중앙값 대비 비율 기반 자릿수 오타 자동 보정"""
    df_c = df.copy()
    median_dict = df_c.groupby('detailed_nm')['adj_price'].median().to_dict()

    def correct_logic(row):
        price  = row['adj_price']
        median = median_dict.get(row['detailed_nm'], price)
        if median == 0 or pd.isna(price):
            return price, '유지(계산불가)'
        ratio = price / median

        if   0.05 <= ratio <= 0.15:  return price * 10,  '×10 보정'
        elif 0.005 <= ratio <= 0.02: return price * 100, '×100 보정'
        elif 5.0 <= ratio <= 15.0:   return price / 10,  '÷10 보정'
        elif 50.0 <= ratio <= 150.0: return price / 100, '÷100 보정'
        else:                         return price,        '유지'

    results = df_c.apply(correct_logic, axis=1)
    df_c['adj_price']       = [r[0] for r in results]
    df_c['correction_type'] = [r[1] for r in results]
    return df_c

df_clean = auto_correct_price_typos(df_clean)

print("✅ 자릿수 오타 보정 결과")
print(df_clean['correction_type'].value_counts().to_string())

In [ ]:
# IQR 기반 잔존 이상치 제거
def remove_iqr_outliers(df):
    """상세품목명 기준 IQR 이탈 데이터 제거 (중앙값 ±20% 최소 버퍼 적용)"""
    stats_df = df.groupby('detailed_nm', as_index=False).agg(
        median=('adj_price', 'median'),
        q1    =('adj_price', lambda x: x.quantile(0.25)),
        q3    =('adj_price', lambda x: x.quantile(0.75)),
    )
    stats_df['iqr']     = stats_df['q3'] - stats_df['q1']
    stats_df['lower_b'] = stats_df['q1'] - np.maximum(1.5 * stats_df['iqr'], stats_df['median'] * 0.2)
    stats_df['upper_b'] = stats_df['q3'] + np.maximum(1.5 * stats_df['iqr'], stats_df['median'] * 0.2)

    df_m     = df.merge(stats_df[['detailed_nm', 'lower_b', 'upper_b']], on='detailed_nm', how='left')
    cond_iqr = (df_m['adj_price'] < df_m['lower_b']) | (df_m['adj_price'] > df_m['upper_b'])
    df_out   = df_m[~cond_iqr].drop(columns=['lower_b', 'upper_b'])

    print(f"✅ IQR 이상치 제거: {len(df_m):,}건 → {len(df_out):,}건  (제거: {cond_iqr.sum():,}건)")
    return df_out

df_clean = remove_iqr_outliers(df_clean)

### 품목별 대 / 중 / 소분류 체계 추가

> 📋 품목 분류 참고: [품목_분류_체계.md](../품목_분류_체계.md)

In [ ]:
# 품목 분류 체계 (대분류 / 중분류 / 소분류)
category_dict = {
    # ── 농축수산물 ─────────────────────────────────────────────
    '쌀':('농축수산물','농산물','곡류'), '콩':('농축수산물','농산물','곡류'),
    '팥':('농축수산물','농산물','곡류'),
    '배추':('농축수산물','농산물','채소'), '무':('농축수산물','농산물','채소'),
    '상추':('농축수산물','농산물','채소'), '대파':('농축수산물','농산물','채소'),
    '양파':('농축수산물','농산물','채소'), '오이':('농축수산물','농산물','채소'),
    '애호박':('농축수산물','농산물','채소'), '깻잎':('농축수산물','농산물','채소'),
    '당근':('농축수산물','농산물','채소'), '파프리카':('농축수산물','농산물','채소'),
    '시금치':('농축수산물','농산물','채소'), '토마토':('농축수산물','농산물','채소'),
    '깐마늘':('농축수산물','농산물','채소'), '마늘':('농축수산물','농산물','채소'),
    '감자':('농축수산물','농산물','채소'), '고구마':('농축수산물','농산물','채소'),
    '콩나물':('농축수산물','농산물','채소'), '버섯':('농축수산물','농산물','채소'),
    '풋고추':('농축수산물','농산물','채소'), '청양고추':('농축수산물','농산물','채소'),
    '꽈리고추':('농축수산물','농산물','채소'), '붉은고추':('농축수산물','농산물','채소'),
    '양배추':('농축수산물','농산물','채소'), '방울토마토':('농축수산물','농산물','채소'),
    '브로콜리':('농축수산물','농산물','채소'), '부추':('농축수산물','농산물','채소'),
    '쪽파':('농축수산물','농산물','채소'), '가지':('농축수산물','농산물','채소'),
    '생강':('농축수산물','농산물','채소'), '미나리':('농축수산물','농산물','채소'),
    '도라지':('농축수산물','농산물','채소'), '갓':('농축수산물','농산물','채소'),
    '열무':('농축수산물','농산물','채소'), '호박':('농축수산물','농산물','채소'),
    '파':('농축수산물','농산물','채소'),
    '사과':('농축수산물','농산물','과일'), '배':('농축수산물','농산물','과일'),
    '바나나':('농축수산물','농산물','과일'), '참외':('농축수산물','농산물','과일'),
    '수박':('농축수산물','농산물','과일'), '오렌지':('농축수산물','농산물','과일'),
    '귤':('농축수산물','농산물','과일'), '단감':('농축수산물','농산물','과일'),
    '감':('농축수산물','농산물','과일'), '포도':('농축수산물','농산물','과일'),
    '딸기':('농축수산물','농산물','과일'), '골드키위':('농축수산물','농산물','과일'),
    '복숭아':('농축수산물','농산물','과일'), '대추':('농축수산물','농산물','과일'),
    '밤':('농축수산물','농산물','과일'),
    '소고기':('농축수산물','축산물','정육'), '돼지고기':('농축수산물','축산물','정육'),
    '닭고기':('농축수산물','축산물','정육'), '계란':('농축수산물','축산물','알류'),
    '고등어':('농축수산물','수산물','생선류'), '갈치':('농축수산물','수산물','생선류'),
    '명태':('농축수산물','수산물','생선류'), '조기':('농축수산물','수산물','생선류'),
    '오징어':('농축수산물','수산물','해산물'), '새우':('농축수산물','수산물','해산물'),
    '조개':('농축수산물','수산물','해산물'), '굴':('농축수산물','수산물','해산물'),
    '낙지':('농축수산물','수산물','해산물'), '전복':('농축수산물','수산물','해산물'),
    '꽃게':('농축수산물','수산물','해산물'),
    '마른멸치':('농축수산물','수산물','건어물/해조류'),
    '맛김':('농축수산물','수산물','건어물/해조류'),
    # ── 가공식품 ───────────────────────────────────────────────
    '설탕':('가공식품','조미료','소스/오일'), '식초':('가공식품','조미료','소스/오일'),
    '식용유':('가공식품','조미료','소스/오일'), '간장':('가공식품','조미료','소스/오일'),
    '마요네즈':('가공식품','조미료','소스/오일'), '케찹':('가공식품','조미료','소스/오일'),
    '참기름':('가공식품','조미료','소스/오일'),
    '새우젓':('가공식품','조미료','젓갈류'), '멸치액젓':('가공식품','조미료','젓갈류'),
    '고춧가루':('가공식품','조미료','가루/장류'), '된장':('가공식품','조미료','가루/장류'),
    '고추장':('가공식품','조미료','가루/장류'), '밀가루':('가공식품','조미료','가루/장류'),
    '부침가루':('가공식품','조미료','가루/장류'),
    '굵은소금':('가공식품','조미료','가루/장류'), '소금':('가공식품','조미료','가루/장류'),
    '라면':('가공식품','면/빵류','면류'), '컵라면':('가공식품','면/빵류','면류'),
    '국수':('가공식품','면/빵류','면류'), '빵':('가공식품','면/빵류','빵류'),
    '통조림':('가공식품','간편식','간편조리'), '즉석밥':('가공식품','간편식','간편조리'),
    '어묵':('가공식품','간편식','간편조리'), '만두':('가공식품','간편식','간편조리'),
    '김치':('가공식품','간편식','간편조리'),
    '햄':('가공식품','간편식','가공육'), '소시지':('가공식품','간편식','가공육'),
    '두부':('가공식품','간편식','두부류'),
    '우유':('가공식품','유제품/간식','유제품'), '치즈':('가공식품','유제품/간식','유제품'),
    '분유':('가공식품','유제품/간식','유제품'), '전지분유':('가공식품','유제품/간식','유제품'),
    '에너지바':('가공식품','유제품/간식','간식류'),
    '초콜릿':('가공식품','유제품/간식','간식류'), '캔디':('가공식품','유제품/간식','간식류'),
    # ── 음료/주류 ──────────────────────────────────────────────
    '사이다':('음료/주류','음료','탄산/생수'), '콜라':('음료/주류','음료','탄산/생수'),
    '생수':('음료/주류','음료','탄산/생수'),
    '소주':('음료/주류','주류','주류'), '맥주':('음료/주류','주류','주류'),
    # ── 생필품 ────────────────────────────────────────────────
    '비누':('생필품','위생용품','바디/헤어'), '샴푸':('생필품','위생용품','바디/헤어'),
    '바디워시':('생필품','위생용품','바디/헤어'),
    '칫솔':('생필품','위생용품','구강용품'), '치약':('생필품','위생용품','구강용품'),
    '세제':('생필품','생활잡화','세제/세정'), '주방세제':('생필품','생활잡화','세제/세정'),
    '섬유유연제':('생필품','생활잡화','세제/세정'),
    '위생백':('생필품','생활잡화','주방잡화'), '고무장갑':('생필품','생활잡화','주방잡화'),
    '일회용컵':('생필품','생활잡화','주방잡화'), '일회용접시':('생필품','생활잡화','주방잡화'),
    '일회용마스크':('생필품','위생용품','위생용품'),
    '휴지':('생필품','위생용품','지류/물티슈'), '물티슈':('생필품','위생용품','지류/물티슈'),
    '기저귀':('생필품','위생용품','위생용품'), '여성용품':('생필품','위생용품','위생용품'),
}

df_category = (
    pd.DataFrame.from_dict(category_dict, orient='index',
                            columns=['대분류', '중분류', '소분류'])
    .reset_index()
    .rename(columns={'index': 'base_nm'})
)

df_clean['base_nm'] = df_clean['detailed_nm'].str.split('_').str[0]
df_clean = df_clean.merge(df_category, on='base_nm', how='left')
df_clean[['대분류', '중분류', '소분류']] = df_clean[['대분류', '중분류', '소분류']].fillna('기타')
df_clean.drop(columns=['base_nm'], inplace=True)

print("✅ 분류 체계 매핑 완료")
print(df_clean.groupby('대분류').size().rename('건수').to_string())

### 데이터셋 깔끔하게

In [ ]:
# 불필요 칼럼 제거 및 칼럼명 한글 통일
drop_cols = ['is_valid', 'correction_type']
drop_cols = [c for c in drop_cols if c in df_clean.columns]
df_clean.drop(columns=drop_cols, inplace=True)

re_col_mapping = {
    'sn':              '일련번호',
    'mkplc_mart_no':   '시장마트번호',
    'mkplc_mart_nm':   '시장마트명',
    'prdlst_no':       '품목번호',
    'prdlst_nm':       '품목명',
    'real_sle_stndrd': '실제판매규격',
    'pc':              '가격',
    'ym':              '연월',
    'rmrk':            '비고',
    'mkplc_type_cd':   '시장유형코드',
    'mkplc_type_nm':   '시장유형명',
    'atdrc_cd':        '자치구코드',
    'atdrc':           '자치구',
    'chck_ymd':        '점검일자',
    'new_std_nm':      '대표품목명',
    'detailed_nm':     '상세품목명',
    'adj_price':       '보정가격',
}

rename_map = {k: v for k, v in re_col_mapping.items() if k in df_clean.columns}
df_clean.rename(columns=rename_map, inplace=True)

print("✅ 전처리 완료")
print(f"   최종 데이터: {df_clean.shape[0]:,}행 × {df_clean.shape[1]}열")
print(f"   분석 기간  : {df_clean['점검일자'].min().strftime('%Y-%m-%d')} ~ {df_clean['점검일자'].max().strftime('%Y-%m-%d')}")
print(f"   자치구 수  : {df_clean['자치구'].nunique()}개")
print(f"   대표 품목  : {df_clean['대표품목명'].nunique()}개")